# ✨ Làm nét ảnh câu chuyện Kanji (Real-ESRGAN anime, ×2)

Tải ảnh `kanji_story/NNNN.webp` từ trang web → phóng ×4 bằng AI rồi thu về ×2 → nén `kanji_story_x2.zip` để tải về.
*Runtime → Change runtime type → T4 GPU* → **Run all**. Khoảng 20–40 phút cho ~2.150 ảnh.

Tải zip về máy, giải nén vào `build/kanji_story_x2/` (ghi đè / gộp với ảnh đã có).

In [ ]:
import torch
assert torch.cuda.is_available(), "⚠️ Chưa có GPU: Runtime → Change runtime type → T4 GPU"
!pip -q install spandrel
!wget -q -nc https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth

In [ ]:
import os, io, time, urllib.request, urllib.error
import numpy as np
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from spandrel import ModelLoader

SITE = "https://testervippro.github.io/Giai-Thich-Ngu-Phap-Tieng-Anh/kanji-viewer/kanji_story/"
SRC, OUT = "/content/src", "/content/kanji_story_x2"
os.makedirs(SRC, exist_ok=True); os.makedirs(OUT, exist_ok=True)
SKIP = set()   # số thẻ đã làm trên máy (bỏ qua cho nhanh), vd set(range(1, 223))

def fetch(n):
    p = f"{SRC}/{n:04d}.webp"
    if os.path.exists(p): return p
    try:
        open(p, "wb").write(urllib.request.urlopen(SITE + f"{n:04d}.webp", timeout=30).read()); return p
    except urllib.error.HTTPError:
        return None   # thẻ không có ảnh
with ThreadPoolExecutor(8) as ex:
    files = [p for p in ex.map(fetch, [n for n in range(1, 2200) if n not in SKIP]) if p]
print(len(files), "ảnh")

In [ ]:
model = ModelLoader().load_from_file("RealESRGAN_x4plus_anime_6B.pth").cuda().eval().half()

@torch.no_grad()
def upscale(im, scale=2, tile=512, pad=16):
    x = torch.from_numpy(np.array(im.convert("RGB"))).permute(2, 0, 1)[None].half().div(255).cuda()
    _, _, h, w = x.shape; s = model.scale
    out = torch.zeros(1, 3, h * s, w * s, device="cuda", dtype=x.dtype)
    for y0 in range(0, h, tile):
        for x0 in range(0, w, tile):
            ya, yb = max(y0 - pad, 0), min(y0 + tile + pad, h); xa, xb = max(x0 - pad, 0), min(x0 + tile + pad, w)
            o = model(x[:, :, ya:yb, xa:xb]); ty, tx = min(y0 + tile, h), min(x0 + tile, w)
            out[:, :, y0*s:ty*s, x0*s:tx*s] = o[:, :, (y0-ya)*s:(ty-ya)*s, (x0-xa)*s:(tx-xa)*s]
    big = Image.fromarray(out[0].float().clamp(0, 1).mul(255).round().byte().permute(1, 2, 0).cpu().numpy())
    return big.resize((w * scale, h * scale), Image.LANCZOS)

t0 = time.time()
for i, p in enumerate(files, 1):
    dst = f"{OUT}/{os.path.basename(p)}"
    if not os.path.exists(dst):
        upscale(Image.open(p)).save(dst, quality=90, method=6)
    if i % 100 == 0: print(f"{i}/{len(files)}  {(time.time() - t0) / i:.2f} s/ảnh")
print("xong", len(os.listdir(OUT)), "ảnh")

In [ ]:
!cd /content && zip -q -r kanji_story_x2.zip kanji_story_x2 && ls -la kanji_story_x2.zip
from google.colab import files
files.download("/content/kanji_story_x2.zip")